#### Simple Gen AI APP Using Langchain

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [2]:
## Data ingestion - we will scrap the data from the website to get the data.

from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
loader = WebBaseLoader("https://www.langchain.com/deep-agents")
loader

In [4]:
docs = loader.load()
docs

[Document(metadata={'source': 'https://www.langchain.com/deep-agents', 'title': 'Deep Agents: Open Source Agent Harness | LangChain', 'description': 'Deep Agents handles planning, context management, and subagent orchestration, so agents can run long, complex work like research and coding.', 'language': 'en'}, page_content='Deep Agents: Open Source Agent Harness | LangChain\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nLangChain has raised a $300M Series C at a $3B valuation|Learn more ↗\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nProducts\n\nLangSmith Platform\n\nAgent ImprovementEngineImprove agents autonomouslyObservabilitySee exactly what your agents are doingEvaluationScore and improve agent performanceAgent RuntimeAgent ServerShip and scale agents in productionSandboxesRun agent-generated code safelyFleetAgents for the whole companyAgent GovernanceLLM GatewayControl agent model callsOpen Source\n\nAgent FrameworksdeepagentsBuild long-running agents for complex taskslanggr

In [5]:
## now we wil divide the data into the chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap=200)


In [6]:
documents = text_splitter.split_documents(docs)

In [7]:
documents

[Document(metadata={'source': 'https://www.langchain.com/deep-agents', 'title': 'Deep Agents: Open Source Agent Harness | LangChain', 'description': 'Deep Agents handles planning, context management, and subagent orchestration, so agents can run long, complex work like research and coding.', 'language': 'en'}, page_content='Deep Agents: Open Source Agent Harness | LangChain\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nLangChain has raised a $300M Series C at a $3B valuation|Learn more ↗\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nProducts\n\nLangSmith Platform\n\nAgent ImprovementEngineImprove agents autonomouslyObservabilitySee exactly what your agents are doingEvaluationScore and improve agent performanceAgent RuntimeAgent ServerShip and scale agents in productionSandboxesRun agent-generated code safelyFleetAgents for the whole companyAgent GovernanceLLM GatewayControl agent model callsOpen Source\n\nAgent FrameworksdeepagentsBuild long-running agents for complex taskslanggr

In [8]:
from langchain_openai import OpenAIEmbeddings 
embeddings = OpenAIEmbeddings()

d:\Learning\Generative AI\Langchain\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
from langchain_community.vectorstores import FAISS
vectorstoredb = FAISS.from_documents(documents, embeddings)

In [10]:
vectorstoredb

In [11]:
query = "Agents are taking on increasingly complex work over long time horizons, like research, coding, and multi-step workflows."
result=vectorstoredb.similarity_search(query)
result[0].page_content

'Get a demo\n\nTry LangSmith\n\nGet a demo\n\n\ndeepagentsBuild agents for complex, multi-step tasksDeep Agents is an open source agent harness built for long-running tasks. It handles planning, context management, and multi-agent orchestration for complex work like research and coding.Start building\n\nRead the docs\n\nWhy use Deep Agents?Designed for autonomous agentsAgents are taking on increasingly complex work over long time horizons, like research, coding, and multi-step workflows. Deep Agents provides the primitives for these patterns:Break down complex objectives: Planning tools let agents decompose tasks, track progress, and adapt as they learnDelegate work in parallel: Spawn subagents for independent subtasks, each with isolated contextPersist knowledge across sessions: Virtual filesystem stores system prompts, skills, and long-term memoryLearn about the agent harness'

In [12]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model= "gpt-4o")


In [13]:
## Retrieval Chain, Document chain


In [ ]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """ 
Answer the following question based only on the provided context:
<context>
{context} ## the context feteched from the vector db will be get stored here.. but in testing of document chain we directly passed the context.
</context>

Question: {input}
    """
)

document_chain=create_stuff_documents_chain(llm,prompt)
document_chain


RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template=' \nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n    '), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000021CE010AD70>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000021CE010B580>, root_client=<openai.OpenAI object at 0x0000021CE0109540>, root_async_client=<openai.AsyncOpenAI object at 0x0000021CE0109870>, model_name='gpt-4o', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_proxy=None, stream_usage=True

In [ ]:
## testing of document chain....this input and context is not passed into  retriever chain...
## the fresh document chain is passed..this is just for testing.
from langchain_core.documents import Document
document_chain.invoke({
    "input":"Agents are taking on increasingly",
    "context":[Document(page_content="Agents are taking on increasingly. complex work over long time horizons, like research, coding, and multi-step workflows. Deep Agents provides the primitives for these patterns: ")]
})

'Deep Agents provides the primitives for patterns involving complex work over long time horizons, such as research, coding, and multi-step workflows.'

In [16]:
### Input--->Retriever--->vectorstoredb

vectorstoredb


In [24]:
retriever=vectorstoredb.as_retriever()
from langchain.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)


In [25]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000021CDFD3C3A0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template=' \nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n    '), additional_kwargs={})])
            |

In [ ]:
## Get the response form the LLM
response=retrieval_chain.invoke({"input":"what is deep agents?"})
response['answer']

'Based on the provided context, Deep Agents is designed for building autonomous agents that can handle complex, non-deterministic, and long-running tasks. It includes tools for planning, context management, and multi-agent orchestration and is suitable for tasks like research and coding. The framework allows agents to decompose tasks, delegate work via subagents, and store knowledge across sessions. Deep Agents is a good choice when you need to build sophisticated, long-running agents, whereas LangGraph is preferred for more low-level control over stateful workflows, and LangChain is ideal for quick starts with standard agent patterns.'

In [29]:

response

{'input': 'what is deep agents?',
 'context': [Document(id='73850d87-c40c-4e80-ba38-0ab4ef8a0bc0', metadata={'source': 'https://www.langchain.com/deep-agents', 'title': 'Deep Agents: Open Source Agent Harness | LangChain', 'description': 'Deep Agents handles planning, context management, and subagent orchestration, so agents can run long, complex work like research and coding.', 'language': 'en'}, page_content='Native context managementContext management is critical for long-running agents, and hard to get right. Deep Agents includes middleware that helps agents compress conversation history, offload large tool results, isolate context with subagents, and use prompt caching to reduce latency and cost.Context management with deep agents\n\nModel neutral with maximum configurabilityDeep Agents is a batteries-included, general purpose agent harness. Use any model provider, manage state, and add human-in-the-loop when you need it. Tracing and deployment work natively with LangSmith.Deploy 

In [30]:
response['context']

[Document(id='73850d87-c40c-4e80-ba38-0ab4ef8a0bc0', metadata={'source': 'https://www.langchain.com/deep-agents', 'title': 'Deep Agents: Open Source Agent Harness | LangChain', 'description': 'Deep Agents handles planning, context management, and subagent orchestration, so agents can run long, complex work like research and coding.', 'language': 'en'}, page_content='Native context managementContext management is critical for long-running agents, and hard to get right. Deep Agents includes middleware that helps agents compress conversation history, offload large tool results, isolate context with subagents, and use prompt caching to reduce latency and cost.Context management with deep agents\n\nModel neutral with maximum configurabilityDeep Agents is a batteries-included, general purpose agent harness. Use any model provider, manage state, and add human-in-the-loop when you need it. Tracing and deployment work natively with LangSmith.Deploy deep agents with LangSmith\n\nBuild with dcode